In [ ]:
import os
import sys
from pathlib import Path

if Path('/kaggle/input').exists():
    print('[INFO] Kaggle Environment Detected. Cloning repository...')
    os.system('git clone https://github.com/dvydinh/smpPrediction.git /kaggle/working/repo')
    sys.path.append('/kaggle/working/repo/v2')
    print('[INFO] Repository cloned and added to sys.path.')


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import get_data_paths, load_and_preprocess_data
from src.feature_engineering import add_engineered_features
from src.model_utils import prepare_daily_matrices, create_median_baseline, flatten_for_global_model, split_and_train_lgb
from src.evaluation import evaluate_and_plot

DATA_ROOT, OUTPUT_DIR = get_data_paths()
print(f'[INFO] DATA_ROOT: {DATA_ROOT}')


In [ ]:
df = load_and_preprocess_data(DATA_ROOT)


In [ ]:
df = add_engineered_features(df)


In [ ]:
X_daily, Y_daily, dates_arr, feature_cols = prepare_daily_matrices(df)
X_daily, Y_daily, Y_base, Y_res, dates_arr = create_median_baseline(X_daily, Y_daily, dates_arr)
X_flat, Y_daily_flat, Y_base_flat, Y_res_flat, dates_flat, ext_feature_cols = flatten_for_global_model(X_daily, Y_daily, Y_base, Y_res, dates_arr, feature_cols)


In [ ]:
global_model, X_test, Y_test, Y_base_test = split_and_train_lgb(X_flat, Y_res_flat, Y_base_flat, dates_flat, ext_feature_cols)


In [ ]:
evaluate_and_plot(global_model, X_test, Y_test, Y_base_test, ext_feature_cols, OUTPUT_DIR)
